In [0]:
catalog_name = "cinedata_analytics"
schema_silver = "silver"
schema_bronze = "bronze"

df_metrics = (
    spark.read.table(f"{catalog_name}.{schema_silver}.tb_movies_metrics")
)

In [0]:
df_origem_metricas = spark.read.table(f"{catalog_name}.{schema_bronze}.tb_movies_metrics")

df_silver_metricas = (
    df_origem_metricas
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("popularity", "popularidade")
    .withColumnRenamed("vote_average", "nota_media_tmdb")
    .withColumnRenamed("vote_count", "qtd_votos_tmdb")
    .withColumnRenamed("averageRating", "nota_media_imdb")
    .withColumnRenamed("numVotes", "qtd_votos_imdb")
)

df_silver_metricas = (
    df_silver_metricas
    .withColumn("popularidade", regexp_replace(col("popularidade"), ",", "."))
)

df_silver_metricas = (
    df_silver_metricas
    .withColumn("popularidade", col("popularidade").try_cast("double"))
)

df_silver_metricas = df_silver_metricas.withColumn(
    "popularidade",
    when(col("popularidade") >= 0, col("popularidade")).otherwise(None)
)

In [0]:
# Converter coluna nota_media_tmdb para double
df_silver_metricas = df_silver_metricas.withColumn(
    "nota_media_tmdb",
    col("nota_media_tmdb").try_cast("double")
)

# Verificar se a nota_media_tmdb está entre 0 e 10
df_silver_metricas = df_silver_metricas.withColumn(
    "nota_media_tmdb",
    when(col("nota_media_tmdb").between(0, 10), col("nota_media_tmdb")).otherwise(
        lit(None)
    ),
)

# Converter coluna nota_media_imdb para double
df_silver_metricas = df_silver_metricas.withColumn(
    "nota_media_imdb",
    col("nota_media_imdb").try_cast("double")
)

# Verificar se a nota_media_imdb está entre 0 e 10
df_silver_metricas = df_silver_metricas.withColumn(
    "nota_media_imdb",
    when(col("nota_media_imdb").between(0, 10), col("nota_media_imdb")).otherwise(
        lit(None)
    ),
)

In [0]:
# Converter colunas qtd_votos_tmdb e qtd_votos_imdb para
df_silver_metricas = df_silver_metricas.withColumn(
    "qtd_votos_tmdb",
    col("qtd_votos_tmdb").try_cast("integer")
)

# Verificar se qtd_votos_tmdb e qtd_votos_imdb são maiores ou iguais a zero
df_silver_metricas = df_silver_metricas.withColumn(
    "qtd_votos_imdb",
    col("qtd_votos_imdb").try_cast("integer")
)

# Verificar se qtd_votos_tmdb e qtd_votos_imdb são maiores ou iguais a zero
df_silver_metricas = df_silver_metricas.withColumn(
    "qtd_votos_imdb",
    when(col("qtd_votos_imdb") >= 0, col("qtd_votos_imdb")).otherwise(None)
)

df_silver_metricas = df_silver_metricas.withColumn(
    "qtd_votos_tmdb",
    when(col("qtd_votos_tmdb") >= 0, col("qtd_votos_tmdb")).otherwise(None)
)

# JUSTIFICATIVA:
# A base bruta possui IDs repetidos com linhas concorrentes (uma contendo o dado
# e a outra contendo NULL). O uso de groupBy("id_filme") com max() consolida os registros,
# pois a função max() ignora valores nulos e preserva as métricas preenchidas.
df_silver_metricas_final = df_silver_metricas.groupBy("id_filme").agg(
    max("popularidade").alias("popularidade"),
    max("nota_media_tmdb").alias("nota_media_tmdb"),
    max("qtd_votos_tmdb").alias("qtd_votos_tmdb"),
    max("nota_media_imdb").alias("nota_media_imdb"),
    max("qtd_votos_imdb").alias("qtd_votos_imdb"),
    max("ingestion_datetime").alias("ingestion_datetime"),
)

# Salvar dados
(
    df_silver_metricas_final.write.format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog_name}.{schema_silver}.tb_metricas_engajamento")
)